# Explore critical-word probabilities with Meltemi

Loads the model once, scores the critical word in each stimulus row against its preceding context, and compares conditions within each item.

Approach: tokenize the *full* sentence once (not context and critical word separately), then use the tokenizer's offset mapping to find which tokens belong to the critical word. This avoids boundary retokenization issues that come from concatenating separately tokenized strings.

In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "ilsp/Meltemi-7B-v1.5"
STIMULI_PATH = "stimuli/stimuli_demo.csv"

In [3]:
# Load model and tokenizer once. This is the slow part, run it once per session.
from transformers import BitsAndBytesConfig

quant_config = BitsAndBytesConfig(load_in_4bit=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
)
model.eval()

print(f"Model loaded, device map: {model.hf_device_map}")

## Scoring function

Given a full sentence and the critical word's character span within it, returns:
- `logprob`: summed log probability of the critical word's tokens, conditioned on everything before it
- `surprisal`: -log2(probability), in bits
- `probability`: exp(logprob), the raw probability (useful as the predictability / cloze-proxy measure)

In [ ]:
import math

def score_critical_word(full_sentence, critical_word, tokenizer, model, device):
    # locate the critical word's character span in the full sentence
    start_char = full_sentence.find(critical_word)
    if start_char == -1:
        raise ValueError(f"Critical word '{critical_word}' not found in sentence: {full_sentence}")
    end_char = start_char + len(critical_word)

    encoding = tokenizer(full_sentence, return_offsets_mapping=True, return_tensors="pt")
    input_ids = encoding["input_ids"].to(device)
    offsets = encoding["offset_mapping"][0]

    # find which token indices fall inside the critical word's character span
    critical_token_idxs = [
        i for i, (s, e) in enumerate(offsets.tolist())
        if s < end_char and e > start_char and not (s == 0 and e == 0)  # skip special tokens
    ]
    if not critical_token_idxs:
        raise ValueError(f"No tokens matched the critical word span for: {critical_word}")

    with torch.no_grad():
        outputs = model(input_ids)
    log_probs = torch.log_softmax(outputs.logits[0], dim=-1)

    # log P(token_i | tokens_<i) comes from the logits at position i-1
    total_logprob = 0.0
    for idx in critical_token_idxs:
        token_id = input_ids[0, idx]
        total_logprob += log_probs[idx - 1, token_id].item()

    return {
        "logprob": total_logprob,
        "surprisal_bits": -total_logprob / math.log(2),
        "probability": math.exp(total_logprob),
        "n_tokens": len(critical_token_idxs),
    }

## Quick sanity check on a single item

Before running the full stimuli set, confirm the function behaves sensibly on one hand-picked example.

In [ ]:
test_sentence = "O papús milúse dinatá ston egonó tu ótan [pro] djávaze éna paramýthi."
test_critical_word = "éna paramýthi"

result = score_critical_word(test_sentence, test_critical_word, tokenizer, model, device)
result

## Score the full stimuli set

In [ ]:
stimuli = pd.read_csv(STIMULI_PATH)
stimuli.head()

In [ ]:
records = []
for _, row in stimuli.iterrows():
    scores = score_critical_word(row["full_sentence"], row["critical_word"], tokenizer, model, device)
    records.append({**row.to_dict(), **scores})

results_df = pd.DataFrame(records)
results_df[["item_id", "condition", "referent_bias", "critical_word", "probability", "surprisal_bits", "n_tokens"]]

## Compare conditions within each item

For each item, the congruent conditions should show noticeably higher probability (lower surprisal) than a mismatched or neutral comparison. If probabilities look flat across conditions, that item's manipulation likely isn't landing and is worth revising or dropping.

In [ ]:
pivot = results_df.pivot_table(index="item_id", columns="condition", values="probability")
pivot

In [ ]:
# save results for downstream use (e.g. reading into your R wrangling project)
results_df.to_csv("output/scored_stimuli_demo.csv", index=False)